# Traffic Control System (Advanced)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def traffic_light_algorithm(traffic_vector):
    # Phase Weight Matrix (2x4)
    # Row 0 checks North + South. Row 1 checks East + West.
    W = np.array([
        [1, 1, 0, 0], 
        [0, 0, 1, 1]  
    ])

    # Green Light (Clearance) Matrices (4x4)
    G_NS = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0]
    ])

    G_EW = np.array([
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])
    
    # ALGORITHM STEP 1: Calculate Priority using Matrix Multiplication
    priority = W @ traffic_vector
    
    # ALGORITHM STEP 2: Determine the winner
    if priority[0] >= priority[1]:
        G_active = G_NS
    else:
        G_active = G_EW

    # ALGORITHM STEP 3: Update traffic using Matrix Subtraction and Multiplication
    traffic_vector = traffic_vector - (0.8 * (G_active @ traffic_vector))
    return traffic_vector

# Simulating traffic control over multiple cycles with visualization
print("\n--- Starting Traffic Simulation ---")

# Let's start with a new traffic vector and simulate 10 cycles of traffic lights
sim_traffic = np.array([50.0, 45.0, 30.0, 25.0])

# Lists to store data for plotting
history_north = [sim_traffic[0]]
history_south = [sim_traffic[1]]
history_east = [sim_traffic[2]]
history_west = [sim_traffic[3]]
cycles = [0]

for cycle in range(1, 11):
    sim_traffic = traffic_light_algorithm(sim_traffic)
    
    # Simulate new cars arriving at each intersection during the cycle
    new_arrivals = np.random.randint(2, 12, size=4)
    sim_traffic += new_arrivals
    
    # Record history
    history_north.append(sim_traffic[0])
    history_south.append(sim_traffic[1])
    history_east.append(sim_traffic[2])
    history_west.append(sim_traffic[3])
    cycles.append(cycle)

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(cycles, history_north, label='North', marker='o')
plt.plot(cycles, history_south, label='South', marker='s')
plt.plot(cycles, history_east, label='East', marker='^')
plt.plot(cycles, history_west, label='West', marker='x')

plt.title('Intersection Traffic Volume Over 10 Cycles')
plt.xlabel('Simulation Cycle')
plt.ylabel('Number of Waiting Cars')
plt.legend()
plt.grid(True)
plt.xticks(cycles)
plt.show()

## Gauss-Jordan Elimination: Solving for Fixed Clearances
Instead of a dynamic simulation with random arrivals, let's look at a **fixed traffic network** where we know exactly how many cars are arriving at each lane per cycle. 

We can frame this as a system of linear equations. Suppose we want to find the exact duration of the green lights for different phases ($t_1, t_2, t_3, t_4$) required to clear a very specific incoming fixed load, based on lane capacities. 

Let the system be $A \cdot T = B$, where:
- $A$ is the matrix of lane clearance rates (cars per second) for each phase.
- $T$ is the vector of green light times we want to solve for.
- $B$ is the predefined (fixed) number of cars arriving.

We will use **Gauss-Jordan Elimination** (row reduction) to systematically reduce the augmented matrix $[A | B]$ to its reduced row echelon form (RREF) to find the exact times.

In [ ]:
import numpy as np

# Fixed number of cars arriving per cycle (Predefined)
# [North, South, East, West]
fixed_arrivals = np.array([40.0, 30.0, 50.0, 45.0])

# Let's say we have 4 distinct light phases with different clearance rates (cars/sec)
# Phase 1: North left turn & straight
# Phase 2: South left turn & straight
# Phase 3: East left turn & straight
# Phase 4: West left turn & straight
# Matrix A (4 phases x 4 directions capacities):
A = np.array([
    [2.0, 0.0, 0.0, 0.0],  # Phase 1 clears 2 cars/sec from North
    [0.0, 1.5, 0.0, 0.0],  # Phase 2 clears 1.5 cars/sec from South
    [0.0, 0.0, 2.5, 0.0],  # Phase 3 clears 2.5 cars/sec from East
    [0.0, 0.0, 0.0, 3.0]   # Phase 4 clears 3 cars/sec from West
])

# Augmented Matrix [A | B]
# We transpose fixed_arrivals to act as the B column
augmented_matrix = np.column_stack((A, fixed_arrivals))
print("Initial Augmented Matrix [A | B]:")
print(augmented_matrix)
print("-" * 40)

def gauss_jordan_elimination(aug_matrix):
    """
    Performs Gauss-Jordan row reduction to solve the system.
    """
    mat = aug_matrix.copy()
    rows, cols = mat.shape
    
    for i in range(rows):
        # 1. Make the diagonal element 1 (Divide the row by the diagonal pivot)
        pivot = mat[i][i]
        if pivot != 0:
            mat[i] = mat[i] / pivot
            print(f"Row {i} -> divided by {pivot}:")
            print(np.round(mat, 2))
            print()
            
        # 2. Make all other elements in this column 0
        for j in range(rows):
            if i != j:
                factor = mat[j][i]
                mat[j] = mat[j] - factor * mat[i]
                if factor != 0:
                    print(f"Row {j} -> Row {j} - ({factor} * Row {i}):")
                    print(np.round(mat, 2))
                    print()
                    
    return mat

# Solve for the exact green light timings using Gauss-Jordan
rref_matrix = gauss_jordan_elimination(augmented_matrix)

# Extract the solved times (the last column of the reduced matrix)
solved_times = rref_matrix[:, -1]

print("--- FINAL SOLUTION (Reduced Row Echelon Form) ---")
print(f"Green Light Time needed for North phase: {solved_times[0]:.1f} seconds")
print(f"Green Light Time needed for South phase: {solved_times[1]:.1f} seconds")
print(f"Green Light Time needed for East phase:  {solved_times[2]:.1f} seconds")
print(f"Green Light Time needed for West phase:  {solved_times[3]:.1f} seconds")
print(f"Total cycle length to clear fixed traffic: {np.sum(solved_times):.1f} seconds")